# Baselines and candidate retrieval

This notebook compares leakage-safe retrieval methods on the development target `n-1`, using only history through `n-2`. It does not load `order_products__train` and contains no learned ranker or feature engineering.

## Methods and ordering

- **Global Popularity:** purchase count in visible development history.
- **Personal Repeat:** per-user purchase frequency; ties use most recent purchase and then product id. Recency is only a tie-breaker, not an opaque score.
- **Explore Popularity:** global popularity after removing the user's historical items.
- **Co-visitation Explore:** cosine-normalized basket co-occurrence, `cooccur(i,j) / sqrt(freq(i) * freq(j))`; user score is the sum of similarities from distinct historical items, with seen items removed.
- **Unions:** deduplicated round-robin by component rank. Component scores are never added; source priority breaks same-rank ties and provenance is retained.

In [ ]:
from pathlib import Path
import gc
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Run from the repository root or notebooks/.')
sys.path.insert(0, str(ROOT))

from src.data import load_instacart, resolve_data_dir
from src.evaluation import (
    assert_no_target_in_history, build_evaluation_examples,
    classify_repeat_explore, history_items, retrieval_recall, target_items,
)
from src.retrieval import (
    build_co_visitation, co_visitation_candidates, global_popularity,
    personal_repeat_candidates, popularity_candidates, union_candidates,
)

MAX_K = 100
METRIC_KS = (10, 20, 50, 100)
MAX_NEIGHBORS = 100
USER_BATCH_SIZE = 2_000
DATA_DIR = resolve_data_dir()
sns.set_theme(style='whitegrid', context='notebook')
DATA_DIR

In [ ]:
# Load each large source once. Final labels remain closed.
data = load_instacart(DATA_DIR, tables=('orders', 'order_products__prior'))
orders = data['orders']
prior_items = data['order_products__prior']
examples = build_evaluation_examples(orders)
history = history_items(orders, prior_items, examples, 'ranker')
target_raw = target_items(prior_items, pd.DataFrame(), examples, 'ranker')
target = classify_repeat_explore(history, target_raw)

assert_no_target_in_history(history, examples, 'ranker')
assert set(examples['ranker_target_order_id']).isdisjoint(history['order_id'])
cutoff_check = history[['user_id', 'order_number']].merge(
    examples[['user_id', 'ranker_history_max_order_number']],
    on='user_id', validate='many_to_one'
)
assert (cutoff_check['order_number'] <= cutoff_check['ranker_history_max_order_number']).all()
assert history['order_id'].isin(prior_items['order_id']).all()

evaluation_users = examples['user_id'].to_numpy()
print(f'Users: {len(evaluation_users):,}')
print(f'History rows: {len(history):,}; target rows: {len(target):,}')
print('Final label rows loaded: 0')
del data, prior_items, target_raw, cutoff_check
gc.collect()

In [ ]:
timings = {}
candidate_stats = {}
metric_frames = []

def timed(name, function, *args, **kwargs):
    start = perf_counter()
    result = function(*args, **kwargs)
    timings[name] = perf_counter() - start
    return result

def evaluate(name, candidates):
    assert not candidates.duplicated(['user_id', 'product_id']).any()
    candidate_stats[name] = {
        'rows': len(candidates),
        'memory_mb': candidates.memory_usage(deep=True).sum() / 2**20,
    }
    metrics = retrieval_recall(candidates, target, METRIC_KS)
    metrics.insert(0, 'generator', name)
    metric_frames.append(metrics)
    return metrics

def result_table(frames):
    result = pd.concat(frames, ignore_index=True)
    return result.loc[result['k'].isin([10, 50, 100])].pivot_table(
        index='generator', columns=['segment', 'k'], values='recall'
    ).round(4)

## E1 — Global Popularity vs Personal Repeat

In [ ]:
popularity = timed('fit_global_popularity', global_popularity, history)
global_candidates = timed(
    'retrieve_global_popularity', popularity_candidates,
    evaluation_users, popularity, MAX_K, source='global_popularity'
)
global_metrics = evaluate('Global Popularity', global_candidates)

personal_candidates = timed(
    'retrieve_personal_repeat', personal_repeat_candidates,
    history, MAX_K, source='personal_repeat'
)
personal_metrics = evaluate('Personal Repeat', personal_candidates)
display(result_table([global_metrics, personal_metrics]))
del global_candidates
gc.collect()

## E2a — Explore Popularity and its union with Personal Repeat

In [ ]:
explore_pop_candidates = timed(
    'retrieve_explore_popularity', popularity_candidates,
    evaluation_users, popularity, MAX_K, history=history, exclude_seen=True,
    source='explore_popularity'
)
explore_pop_metrics = evaluate('Explore Popularity', explore_pop_candidates)
repeat_pop_union = timed(
    'union_repeat_popularity', union_candidates,
    [personal_candidates, explore_pop_candidates], MAX_K,
    source_priority=['personal_repeat', 'explore_popularity']
)
repeat_pop_metrics = evaluate('Repeat + Explore Popularity', repeat_pop_union)
display(result_table([explore_pop_metrics, repeat_pop_metrics]))
del explore_pop_candidates, repeat_pop_union
gc.collect()

## E2b — Co-visitation Explore

The sparse order×item product is the expected peak-memory step. The full co-occurrence graph is normalized first, then pruned to `MAX_NEIGHBORS` per item. User scoring is batched and never creates a dense user×item matrix.

In [ ]:
co_visitation = timed(
    'fit_co_visitation', build_co_visitation,
    history, max_neighbors=MAX_NEIGHBORS
)
co_vis_memory_mb = (
    co_visitation.similarity.data.nbytes
    + co_visitation.similarity.indices.nbytes
    + co_visitation.similarity.indptr.nbytes
    + co_visitation.item_ids.nbytes
) / 2**20
pd.Series({
    'items': len(co_visitation.item_ids),
    'retained_similarity_edges': co_visitation.similarity.nnz,
    'retained_index_memory_mb': co_vis_memory_mb,
    'max_neighbors': co_visitation.max_neighbors,
}).to_frame('value')

In [ ]:
co_vis_candidates = timed(
    'retrieve_co_visitation', co_visitation_candidates,
    history, evaluation_users, co_visitation, MAX_K,
    exclude_seen=True, batch_size=USER_BATCH_SIZE,
    source='co_visitation_explore'
)
co_vis_metrics = evaluate('Co-visitation Explore', co_vis_candidates)
repeat_co_vis_union = timed(
    'union_repeat_co_visitation', union_candidates,
    [personal_candidates, co_vis_candidates], MAX_K,
    source_priority=['personal_repeat', 'co_visitation_explore']
)
repeat_co_vis_metrics = evaluate('Repeat + Co-visitation', repeat_co_vis_union)
display(result_table([co_vis_metrics, repeat_co_vis_metrics]))
del co_vis_candidates, repeat_co_vis_union
gc.collect()

## Retrieval results and Recall@K curve

In [ ]:
results = pd.concat(metric_frames, ignore_index=True)
display(result_table(metric_frames))

overall_curve = results.loc[results['segment'].eq('overall')]
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(
    data=overall_curve, x='k', y='recall', hue='generator',
    marker='o', ax=ax
)
ax.set(title='Development candidate Recall@K', xlabel='candidate pool K', ylabel='macro overall recall')
ax.legend(title='generator', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()

In [ ]:
def recall_value(generator, k, segment):
    return results.loc[
        results['generator'].eq(generator)
        & results['k'].eq(k)
        & results['segment'].eq(segment),
        'recall',
    ].item()

explore_delta = (
    recall_value('Co-visitation Explore', 100, 'explore')
    - recall_value('Explore Popularity', 100, 'explore')
)
print('Personal Repeat overall Recall@100:', f"{recall_value('Personal Repeat', 100, 'overall'):.4f}")
print('Personal Repeat repeat Recall@100:', f"{recall_value('Personal Repeat', 100, 'repeat'):.4f}")
print('Co-visitation minus Popularity explore Recall@100:', f'{explore_delta:+.4f}')

hybrids = ['Repeat + Explore Popularity', 'Repeat + Co-visitation']
best_hybrid = (
    results.loc[
        results['generator'].isin(hybrids)
        & results['segment'].eq('overall')
        & results['k'].eq(MAX_K)
    ]
    .sort_values('recall', ascending=False)
    .iloc[0]['generator']
)
hybrid_curve = overall_curve.loc[overall_curve['generator'].eq(best_hybrid)].sort_values('k')
threshold = 0.95 * hybrid_curve.loc[hybrid_curve['k'].eq(MAX_K), 'recall'].item()
recommended_k = int(hybrid_curve.loc[hybrid_curve['recall'].ge(threshold), 'k'].min())
print(f'Transparent K heuristic: {recommended_k} (first K reaching 95% of {best_hybrid} Recall@{MAX_K})')

## Runtime and retained-memory diagnostics

In [ ]:
display(pd.Series(timings, name='seconds').sort_values(ascending=False).to_frame())
candidate_diagnostics = pd.DataFrame(candidate_stats).T.sort_values('memory_mb', ascending=False)
display(candidate_diagnostics.round({'memory_mb': 1}))
print(f'Retained sparse co-visitation index: {co_vis_memory_mb:.1f} MiB')
print('Peak memory during sparse X.T @ X is not captured by retained-object sizes; monitor the Kaggle session during that cell.')